# 01 · Synthetic training scenes

To learn *where* a sign is, the network needs thousands of photos with the sign's position marked. GTSRB has
50,000 sign photos, but each one is a tight crop with the sign filling the frame, so there is nothing to
search. This project makes its own scenes instead: **paste a GTSRB sign crop at a random size and place onto
a COCO photo** (everyday pictures: streets, kitchens, animals). We placed the sign, so we know exactly where it
is.

**You'll see:** where signs and backgrounds come from, how one scene is built step by step, the extra
"augmentation" training scenes get, and the stored datasets.

**Needs:** the raw GTSRB and COCO downloads (sections 2–4, 6) and `data/processed/val.npz` (section 5).
**Writes:** nothing, unless you turn on `REBUILD`.

In [ ]:
# ---- Settings: the only cell you need to edit ----
CONFIG = "configs/peak_fresh_aug.yaml"  # its data: section drives everything below
REBUILD = False  # True: regenerate data/processed/{train,val,test}.npz (a few minutes; same bytes as before)

In [ ]:
# Setup (the same in every notebook). All paths in this project (configs/, data/, outputs/) are relative
# to the repo root, and the scripts expect to run from there, so move this notebook's working folder up
# to the root first. Safe to run twice.
import os
import sys
from pathlib import Path

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists():  # walk up until we reach the folder holding pyproject.toml
    if root == root.parent:
        raise RuntimeError("can't find the repo root: open this notebook from inside FindTheObject-CNN-Project")
    root = root.parent
os.chdir(root)
if str(root / "notebooks") not in sys.path:
    sys.path.insert(0, str(root / "notebooks"))  # lets us import notebooks/nbhelpers.py

from nbhelpers import NB_OUT, ROOT, find_run, read_json, require, run_script
print("working in", ROOT)

In [ ]:
# Stop here, with a clear message, if a file this notebook reads is missing.
require("data/raw/gtsrb/GTSRB/Final_Training/Images", "data/raw/coco/val2017",
        how="download them as described in README.md (Data)")
require("data/processed/val.npz", how="python scripts/make_dataset.py --config configs/baseline.yaml")

## 1. The config

Every setting lives in a YAML file in `configs/`. The `data:` section says how many scenes to make and how to
make them. Its `synth:` part gives the sign size range (16–48 px in a 128 px scene), brightness range, blur and
noise.

In [ ]:
# Load the YAML config into a Python dict and print the scene-making settings.
import yaml

cfg = yaml.safe_load((ROOT / CONFIG).read_text())
data_cfg = cfg["data"]
print("scenes per split:", {k: data_cfg[k] for k in ("n_train", "n_val", "n_test")})
print("synth:", data_cfg["synth"])
print("train_augment:", data_cfg.get("train_augment"))

## 2. Where the signs and backgrounds come from

Three splits, kept strictly apart so that val and test only show the model things it has never seen:

* **Signs.** GTSRB photos come in *tracks*: about 30 frames of the same physical sign as a car drives up to it.
  Two frames from one track are nearly identical. So **train and val are split by whole track**: 10% of each
  class's tracks go to val. Splitting frame by frame would put near-copies of training signs into val. **Test**
  uses GTSRB's official test set, which is separate from all of these.
* **Backgrounds.** COCO photos are dealt out by position in the list: every 5th photo goes to test, the next one
  to val, and the other three to train. Photos that contain a real stop sign are left out, so the only sign
  in a scene is the one we pasted.

In [ ]:
# Build each split's list of sign crops and background photos (reads GTSRB's CSV files and COCO's JSON; ~1 s).
from findobject.data.splits import split_sources

sources = split_sources(data_cfg)
for split, (signs, backgrounds) in sources.items():
    print(f"{split:5}: {len(signs):6,} sign photos, {len(backgrounds):5,} background photos")

## 3. Making one scene, step by step

`make_scene(sign, background, rng, config)` does it all:

1. Cut a random square out of the background photo and resize it to 128×128.
2. Resize the sign so its longer side is a random 16–48 px, and change its brightness at random.
3. Paste it at a random spot. Its box `(x1, y1, x2, y2)` is now known exactly.
4. Maybe blur the whole scene, then add pixel noise, so the pasted sign's edges don't give it away.
5. Build the **target heatmap**: a 32×32 grid with a small Gaussian "bump" at the sign's center. This is what
   the heatmap output is trained to reproduce.

`rng` is a seeded random number generator. The same seed always gives the same scene, which is why rebuilding a
dataset reproduces it byte for byte.

In [ ]:
# Pick one training sign and one training background (seeded, so you see the same pair every time).
import numpy as np
from PIL import Image

from findobject.data.class_names import CLASS_NAMES
from findobject.data.data_config import synth_config
from findobject.data.gtsrb import load_sign
from findobject.data.synth import make_scene

pick = np.random.default_rng(0)
train_signs, train_bgs = sources["train"]
sign_sample = train_signs[int(pick.integers(len(train_signs)))]
sign = load_sign(sign_sample)  # opens the photo and crops it to the sign's box from the CSV
with Image.open(train_bgs[int(pick.integers(len(train_bgs)))]) as f:
    background = f.convert("RGB")

synth = synth_config(data_cfg)  # the YAML synth: section as a typed, checked object
scene = make_scene(sign, background, np.random.default_rng(1), synth)
print("class:", sign_sample.class_id, CLASS_NAMES[sign_sample.class_id], "| sign crop", sign.size,
      "| background", background.size, "| box in scene", scene.box)

In [ ]:
# Show the four pieces: the sign crop, the background, the finished scene with its box, and the target heatmap.
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle


def draw_box(ax, box, scale=1.0, color="lime"):
    """Outline box (x1, y1, x2, y2; x2/y2 exclusive) on ax. scale=1/4 maps scene pixels to heatmap cells."""
    x1, y1, x2, y2 = (v * scale for v in box)
    ax.add_patch(Rectangle((x1 - 0.5, y1 - 0.5), x2 - x1, y2 - y1, fill=False, edgecolor=color, linewidth=1.2))


fig, axes = plt.subplots(1, 4, figsize=(14, 3.6))
axes[0].imshow(sign), axes[0].set_title(f"sign crop {sign.size[0]}x{sign.size[1]}")
axes[1].imshow(background), axes[1].set_title(f"background {background.size[0]}x{background.size[1]}")
axes[2].imshow(scene.image), axes[2].set_title("scene 128x128 + box")
draw_box(axes[2], scene.box)
axes[3].imshow(scene.heatmap, cmap="inferno"), axes[3].set_title("target heatmap 32x32")
draw_box(axes[3], scene.box, scale=1 / 4)  # one heatmap cell = 4x4 scene pixels
for ax in axes:
    ax.set_axis_off()
plt.show()

Every random choice changes from scene to scene: the crop of the background, the sign's size, position and
brightness, the blur and the noise. Here is the **same sign** with eight different seeds:

In [ ]:
# The same sign and background, eight seeds: everything random changes, the sign's class stays the same.
fig, axes = plt.subplots(1, 8, figsize=(16, 2.4))
for seed, ax in enumerate(axes):
    s = make_scene(sign, background, np.random.default_rng(seed), synth)
    ax.imshow(s.image), draw_box(ax, s.box), ax.set_axis_off()
    ax.set_title(f"seed {seed}, {s.box[2] - s.box[0]}px", fontsize=8)
plt.show()

## 4. Augmentation for training scenes only (v1.5b)

From v1.5b on, training signs also get tilted (±10°), warped as if seen at an angle, changed in contrast and
saturation, and sometimes partly covered by a patch of background. The hue never changes, because a sign's
color carries meaning. Val and test scenes **never** get this, so every version is scored on the same
unaugmented scenes.

In [ ]:
# Top row: the plain settings used for val/test. Bottom row: the same seeds with training augmentation on.
# (Positions differ between rows: augmentation draws extra random numbers, so the same seed lands elsewhere.)
from findobject.data.data_config import train_synth_config

train_synth = train_synth_config(data_cfg)  # synth: plus train_augment:
fig, axes = plt.subplots(2, 8, figsize=(16, 4.4))
for seed in range(8):
    for row, (name, c) in enumerate((("plain", synth), ("augmented", train_synth))):
        s = make_scene(sign, background, np.random.default_rng(100 + seed), c)
        axes[row, seed].imshow(s.image), draw_box(axes[row, seed], s.box), axes[row, seed].set_axis_off()
        axes[row, seed].set_title(name, fontsize=8)
plt.show()

## 5. The stored datasets

`make_dataset.py` runs `make_scene` 20,000 times for train and 5,000 times each for val and test, and saves
the results as `.npz` files. `SignScenes` is the PyTorch `Dataset` that serves them. It converts each scene to
the format the network wants: a float tensor `[3, 128, 128]` with values from 0 to 1.

In [ ]:
# Load val.npz and draw a dozen of its scenes with their boxes and class names.
from findobject.data.dataset import SignScenes

val = SignScenes(ROOT / "data/processed/val.npz")
print(len(val), "val scenes; one scene's tensors:", {k: tuple(v.shape) for k, v in val[0]._asdict().items()})

fig, axes = plt.subplots(2, 6, figsize=(15, 5.6))
for i, ax in enumerate(axes.flat):
    s = val[i]
    ax.imshow(s.image.permute(1, 2, 0))  # [3, H, W] -> [H, W, 3] for matplotlib
    draw_box(ax, s.box.tolist()), ax.set_axis_off()
    ax.set_title(CLASS_NAMES[int(s.class_id)], fontsize=8)
plt.show()

In [ ]:
# How many val scenes each class gets. GTSRB has far more speed-limit photos than rare signs, and scenes draw
# signs evenly from the photos, so that imbalance carries over. It's why rare classes have noisier accuracy later.
import collections

counts = collections.Counter(int(val[i].class_id) for i in range(len(val)))
plt.figure(figsize=(14, 2.8))
plt.bar(range(43), [counts[k] for k in range(43)])
plt.xlabel("class id"), plt.ylabel("val scenes"), plt.title("val scenes per class")
plt.show()
print("most:", CLASS_NAMES[max(counts, key=counts.get)], max(counts.values()),
      "| fewest:", CLASS_NAMES[min(counts, key=counts.get)], min(counts.values()))

## 6. Training scenes are made fresh every epoch (v1.5b and later)

v1 and v1.5 reused the same 20,000 stored training scenes every epoch, so the model could start to memorize
them. From v1.5b on, `data.train_scenes: fresh` builds **20,000 brand-new scenes every epoch**, with a seed set
by (seed, epoch). So no scene repeats, yet a rerun still gets exactly the same scenes. `epoch_datasets` returns
a function: give it an epoch number and it gives back that epoch's dataset.

In [ ]:
# The first five training scenes of epoch 1 and of epoch 2: different scenes, but the same every time you run this.
from findobject.training.train_data import epoch_datasets

for_epoch = epoch_datasets(data_cfg, seed=cfg["train"]["seed"])
fig, axes = plt.subplots(2, 5, figsize=(13, 5.6))
for row, epoch in enumerate((1, 2)):
    ds = for_epoch(epoch)
    for i in range(5):
        s = ds[i]
        axes[row, i].imshow(s.image.permute(1, 2, 0)), draw_box(axes[row, i], s.box.tolist())
        axes[row, i].set_title(f"epoch {epoch}, scene {i}", fontsize=8), axes[row, i].set_axis_off()
plt.show()

## Run it for real: rebuild the stored datasets

`make_dataset.py` regenerates `train.npz`, `val.npz` and `test.npz` from the raw downloads. Everything is
seeded, so the new files are byte-for-byte the same as the old ones. All configs share the same `data:` and
`synth:` settings, so any of them will do. (`train_augment` is ignored here: stored scenes are never augmented.)

For more pictures of the data, see `show_data.py` and `show_train_scenes.py` in notebook 08.

In [ ]:
# Regenerate data/processed/{train,val,test}.npz. Only runs when REBUILD = True (Settings cell).
run_script("make_dataset", "--config", CONFIG, enabled=REBUILD)

**Next:** `02_model.ipynb`, the network that learns from these scenes.